# EDA on raw PeMS data: US-101, District 4, January 2017

**Data:** `data/raw/d04_2017_01_station_5min.csv.gz`, downloaded with `src/data/load_pems_data.py`
(`--year 2017 --month 1 --district 4 --highways 101`). Re-run that script if the file is missing.
If you downloaded with `--dow M`, the file holds only the five Mondays.

**Why this notebook exists:** PEMS-BAY gives us speeds only, with gaps already filled in by PeMS.
The raw files add three things we need:
- `% Observed`: how much of each value came from a working detector rather than a PeMS estimate
- `Total Flow`: actual vehicle counts, so we can say which stretches are busiest
- `Lane Type`: so we can keep mainline lanes and drop ramps and HOV lanes

**How to use it:** each section asks one question. Read the hint, write code in the empty cell,
run it with Shift+Enter, then write what you found in a sentence under it.

## 0. Setup and load

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works whether the notebook runs from the repo root or from notebooks/
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"

DATA_FILE = RAW / "d04_2017_01_station_5min.csv.gz"
print(DATA_FILE, "exists:", DATA_FILE.exists())

In [ ]:
df = pd.read_csv(DATA_FILE)
df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%m/%d/%Y %H:%M:%S")

# Shorter column names are easier to type
df = df.rename(columns={
    "Freeway #": "freeway",
    "Direction of Travel": "direction",
    "Lane Type": "lane_type",
    "Station Length": "station_length",
    "% Observed": "pct_observed",
    "Total Flow": "flow",
    "Avg Occupancy": "occupancy",
    "Avg Speed": "speed",
}).rename(columns=str.lower)

df.shape

In [ ]:
df.head()

In [ ]:
# The 33 US-101 southbound mainline sensors in PEMS-BAY, ordered in the direction of travel
# (first = most upstream, postmile 398.3; last = most downstream, postmile 383.7)
CORRIDOR_IDS = [
    401926, 400149, 401937, 401936, 402373, 400435, 404554, 400545, 400743, 400330, 402371,
    404553, 400904, 402369, 400643, 400794, 402367, 400069, 400372, 400206, 400895, 400586,
    400172, 402365, 400394, 400922, 400109, 402363, 400440, 400160, 401891, 401906, 402360,
]
len(CORRIDOR_IDS)

## 1. What is in the file?

**Questions:** Which lane types are there, and how many stations of each? How many days and
timestamps does the file cover? Are there any missing values, and in which columns?

**Hints:** `value_counts()` on a column; `groupby("lane_type")["station"].nunique()` counts distinct
stations per lane type; `df["timestamp"].dt.date.nunique()`; `df.isna().sum()`.

**Check yourself:** there are 288 five-minute intervals in a day. Does the number of timestamps
match the number of days you downloaded?

In [ ]:
# your code here

**Finding:** 

## 2. How much of the data is a PeMS estimate?

**Questions:** What does the distribution of `pct_observed` look like? What fraction of rows are
0% observed (fully estimated)? What fraction are 100% observed?

**Hints:** `describe()`; `(df["pct_observed"] == 0).mean()` gives a fraction, because True counts as 1;
a histogram with `plt.hist(df["pct_observed"], bins=20)`.

**Think about it:** if a value is 0% observed but still has a speed, where did that speed come from,
and should a model be trained on it?

In [ ]:
# your code here

**Finding:** 

## 3. Narrow down to our corridor

**Task:** make a new DataFrame `corr` with only southbound, mainline (`"ML"`) rows for the
stations in `CORRIDOR_IDS`.

**Hints:** combine conditions with `&`, wrapping each in parentheses:
`df[(df["direction"] == "S") & (df["lane_type"] == "ML") & (df["station"].isin(CORRIDOR_IDS))]`.
Add `.copy()` at the end so later changes don't trigger warnings.

**Check yourself:** `corr["station"].nunique()` should be 33. If it's smaller, find which IDs are
missing with `set(CORRIDOR_IDS) - set(corr["station"])`, and check whether they appear in `df` with
a different direction or lane type.

In [ ]:
# your code here

**Finding:** 

## 4. Which corridor sensors are least trustworthy?

**Questions:** For each of the 33 sensors, what is the average `pct_observed`? Which sensors are
mostly estimated? Are the bad sensors clustered along the road or scattered?

**Hints:** `corr.groupby("station")["pct_observed"].mean().sort_values()`. To see them in road order,
use `.reindex(CORRIDOR_IDS)` instead of sorting, then plot it as a bar chart with `.plot.bar()`.

In [ ]:
# your code here

**Finding:** 

## 5. Night vs. rush hour

**Before running anything, write your prediction here:** will `pct_observed` be higher or lower at
5 pm than at 3 am, and why?

**Prediction:**

**Question:** How does the average `pct_observed` on the corridor change by hour of day?

**Hints:** make an hour column with `corr["hour"] = corr["timestamp"].dt.hour`, then
`corr.groupby("hour")["pct_observed"].mean().plot(marker="o")`.

In [ ]:
# your code here

**Finding (was your prediction right?):** 

## 6. Where and when is traffic heaviest?

`flow` is the number of vehicles that passed the station in 5 minutes, summed over all lanes.

**Questions:** What does average flow look like by hour of day on the corridor? Which sensors carry
the most traffic? Does the busiest stretch match the congested stretch from the heatmap
(postmiles about 389 to 393)?

**Hints:** the same `groupby("hour")` pattern as section 5, using `flow`.
For a per-sensor ranking: `corr.groupby("station")["flow"].mean().reindex(CORRIDOR_IDS)`.
Multiply a 5-minute flow by 12 to get vehicles per hour.

**Careful:** compute this twice, once on all rows and once only on rows with `pct_observed == 100`.
If the answers differ a lot, the estimated values are changing the picture.

In [ ]:
# your code here

**Finding:** 

## 7. Speed and flow together (the fundamental diagram)

**Question:** Pick one busy sensor and scatter-plot `flow` (x-axis) against `speed` (y-axis).
What shape do you see?

**Hints:** `plt.scatter(x, y, s=3, alpha=0.3)`. Color the points by whether they are 100% observed
(plot the two groups separately) to see whether the estimates fall in the same places as real data.

**Think about it:** traffic theory says that as traffic gets heavier, flow first rises at nearly
constant speed, then speed collapses and flow drops. Can you find the free-flow branch and the
congested branch in your plot?

In [ ]:
# your code here

**Finding:** 

## 8. Summary for the team

Write three to five bullet points: what surprised you, what it means for the model, and what
we should decide as a team (for example, whether to drop sensors that are mostly estimated, or
to train only on observed values).

-
-
-